> ## SOLUTION / ANSWER KEY &mdash; Lab 4.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-improve-and-evaluate.ipynb`](../lab-11-improve-and-evaluate.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.11 &mdash; Improve & Evaluate Your Fine-tuned Model

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Compare a barely-trained model to a properly-trained one
- Give the fine-tune enough training steps to converge
- Evaluate with a confusion matrix built from real predictions

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Cost / latency / data tradeoffs](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-11")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
A first fine-tune is rarely your best. The **training budget** (how many steps) matters: too few and
the model **underfits** (barely better than chance); enough and it converges. We fine-tune the **real**
bert-tiny with a small budget vs a larger one, then **evaluate** with a confusion matrix of its actual
predictions to see exactly which class it confuses. (More steps help until they overfit &mdash; tuning
is empirical.)

## Build it
Fill in a too-small budget and a sufficient one, then predict for the confusion matrix.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def train_eval(Xtr, ytr, Xval, yval, steps, lr=5e-3):
    tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
    torch.manual_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)
    enc = tok(Xtr, padding=True, truncation=True, return_tensors="pt"); yt = torch.tensor(ytr)
    opt = torch.optim.AdamW(model.parameters(), lr=lr); model.train()
    for _ in range(steps):
        opt.zero_grad(); out = model(**enc, labels=yt); out.loss.backward(); opt.step()
    model.eval()
    venc = tok(Xval, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad(): pred = model(**venc).logits.argmax(-1).numpy()
    return float((pred == np.array(yval)).mean()), pred

def weak(Xtr, ytr, Xval, yval):
    return train_eval(Xtr, ytr, Xval, yval, steps=2)[0]
def strong(Xtr, ytr, Xval, yval):
    return train_eval(Xtr, ytr, Xval, yval, steps=40)

## Run it for real
Fine-tune with each budget and read the confusion matrix of real predictions.

In [ ]:
try:
    # A tiny labelled sentiment dataset (1 = positive, 0 = negative)
    SENT = [
        ("i love this it is great", 1), ("a great and wonderful film", 1),
        ("truly wonderful i love it", 1), ("excellent and brilliant work", 1),
        ("the best most brilliant story", 1), ("i love how great it is", 1),
        ("wonderful excellent and great fun", 1), ("a brilliant and great success", 1),
        ("great fun i really love it", 1), ("the best film wonderful and brilliant", 1),
        ("excellent great and lovely work", 1), ("i love this brilliant great film", 1),
        ("wonderful great and the best", 1), ("so good i love it great", 1),
        ("i hate this it is terrible", 0), ("a terrible and awful film", 0),
        ("truly awful i hate it", 0), ("boring and terrible work", 0),
        ("the worst most boring story", 0), ("i hate how bad it is", 0),
        ("awful boring and dull mess", 0), ("a terrible and bad failure", 0),
        ("boring mess i really hate it", 0), ("the worst film awful and boring", 0),
        ("terrible bad and dull work", 0), ("i hate this awful boring film", 0),
        ("awful terrible and the worst", 0), ("so bad i hate it terrible", 0),
    ]
    texts  = [t for t, y in SENT]
    labels = [y for t, y in SENT]
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix
    Xtr, Xval, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)

    weak_acc = weak(Xtr, ytr, Xval, yval)
    strong_acc, preds = strong(Xtr, ytr, Xval, yval)
    print("under-trained (2 steps) val acc:", round(weak_acc, 3))
    print("well-trained  (40 steps) val acc:", round(strong_acc, 3))
    cm = confusion_matrix(yval, preds)
    print("confusion matrix [[TN,FP],[FN,TP]] for the well-trained model:")
    print(cm)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    plt.imshow(cm, cmap="Blues"); plt.title("Confusion matrix (fine-tuned bert-tiny)"); plt.colorbar()
    plt.xlabel("predicted"); plt.ylabel("true"); plt.tight_layout()
    plt.savefig(WORK + "/confusion.png", dpi=90)
    print("saved:", WORK + "/confusion.png")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- **2 steps** barely moves off chance &mdash; the model **underfits**. A **sufficient budget** converges to high accuracy. Same model, same data: only the training budget changed.
- The **confusion matrix** shows exactly which class the well-trained model confuses (if any) &mdash; far more informative than a lone accuracy number.
- Real iteration is this loop: change one knob (steps, lr), re-measure on held-out data, read the confusion matrix. Small honest steps.

## Your turn (open task &mdash; no grader)
Sweep `steps` (2, 5, 10, 20, 40) and plot accuracy &mdash; where does it plateau? Push `lr`
too high (e.g. `1e-1`) and watch it destabilise. A "good" answer: you can name the training budget
where accuracy stops improving, and describe one sign of over- vs under-training.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    from sklearn.model_selection import train_test_split
    Xtr, Xval, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
    # sweep the training budget and watch accuracy climb then plateau
    for steps in [2, 5, 10, 20, 40]:
        acc, _ = train_eval(Xtr, ytr, Xval, yval, steps=steps)
        print(f"steps={steps:2d}: val acc = {round(acc, 3)}")
    hi, _ = train_eval(Xtr, ytr, Xval, yval, steps=40, lr=1e-1)   # lr too high -> destabilises
    print("lr=1e-1 (too high) val acc:", round(hi, 3), "-- underfitting sign: acc stuck near chance; overfitting sign: train perfect but val drops.")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
Iterate: change the training budget, re-measure on held-out data, read the confusion matrix. That is the real fine-tuning loop &mdash; and 'more steps' is not always better.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>